# Deep Learning Project 1 — ResNet under 5M parameters on CIFAR-10

**Team:** _fill in names_  ·  **Repo:** _fill in GitHub link_

This notebook reproduces the full pipeline end-to-end and shows the code execution:
1. Build the model and **print the parameter count**
2. Train from scratch (or load the checkpoint produced by `train.py`)
3. **Print the final test accuracy**
4. Plot training curves, confusion matrix, and ablations
5. Generate the Kaggle `submission.csv`

> Run on a GPU (Colab / NYU Greene). On a GPU this notebook does *not* retrain by default — it loads
> `checkpoints/<run>/best.pt` created by `train.py` so that the TA can verify results in minutes.
> Set `RETRAIN = True` to train inside the notebook.

In [ ]:
import os, sys, json, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('..'))          # notebook lives in notebooks/
os.chdir('..')
from models.resnet import build_model, count_parameters, CONFIGS
from data import get_cifar10_loaders, CLASSES
from train import evaluate
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__)

## 1. Model & parameter count
All candidate architectures we considered, with their parameter counts. The budget is **5,000,000**.

In [ ]:
rows = []
for name in CONFIGS:
    n = count_parameters(build_model(name))
    rows.append({'config': name, **CONFIGS[name], 'params': n, 'under_5M': n <= 5_000_000})
pd.DataFrame(rows)

In [ ]:
MODEL = 'resnet_se_343'
RUN_DIR = f'checkpoints/{MODEL}_s0'
model = build_model(MODEL).to(device)
n_params = count_parameters(model)
print(f'Model: {MODEL}')
print(f'NUMBER OF TRAINABLE PARAMETERS: {n_params:,}')
assert n_params <= 5_000_000
# torchsummary-style view (optional): pip install torchinfo
try:
    from torchinfo import summary; summary(model, (1, 3, 32, 32), depth=2)
except ImportError:
    print(model)

## 2. Train (or load checkpoint)

In [ ]:
RETRAIN = False
if RETRAIN:
    !python train.py --model {MODEL} --epochs 200 --batch_size 128 --lr 0.1 --train_csv data/train_labels.csv --out {RUN_DIR}
ck = torch.load(f'{RUN_DIR}/best.pt', map_location=device)
model.load_state_dict(ck['ema'] if 'ema' in ck else ck['model'])
model.eval()
print('loaded epoch', ck['epoch'] + 1)

## 3. Final test accuracy

In [ ]:
_, _, test_loader = get_cifar10_loaders('./data', batch_size=256, num_workers=2)
acc, loss = evaluate(model, test_loader, device)
acc_tta, _ = evaluate(model, test_loader, device, tta=True)
print('=' * 60)
print(f'FINAL TEST ACCURACY:              {acc:.2f}%')
print(f'FINAL TEST ACCURACY (+flip TTA):  {acc_tta:.2f}%')
print(f'NUMBER OF PARAMETERS:             {n_params:,}')
print('=' * 60)

## 4. Training curves

In [ ]:
df = pd.read_csv(f'{RUN_DIR}/log.csv')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(df.epoch, df.train_loss, label='train'); ax[0].plot(df.epoch, df.test_loss, label='test'); ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(df.epoch, df.test_acc, label='raw weights'); ax[1].plot(df.epoch, df.ema_test_acc, label='EMA weights'); ax[1].set_ylim(80, 100); ax[1].grid(alpha=.3); ax[1].set_title('Test accuracy (%)'); ax[1].legend()
ax[2].plot(df.epoch, df.lr); ax[2].set_title('Learning rate')
for a in ax: a.set_xlabel('epoch')
plt.tight_layout(); plt.savefig('figures/curves.png', dpi=150); plt.show()
print('best EMA test acc: %.2f%% at epoch %d' % (df.ema_test_acc.max(), df.ema_test_acc.idxmax() + 1))

## 5. Confusion matrix & per-class accuracy

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
preds, labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(model(x.to(device)).argmax(1).cpu()); labels.append(y)
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout(); plt.savefig('figures/confusion.png', dpi=150); plt.show()
print(pd.Series(cm.diagonal() / cm.sum(1) * 100, index=CLASSES).round(2).rename('per-class acc %'))

## 6. Ablation summary
Fill this table from the `final.json` of each run in `checkpoints/`.

In [ ]:
rows = []
for d in sorted(os.listdir('checkpoints')):
    f = f'checkpoints/{d}/final.json'
    if os.path.exists(f):
        r = json.load(open(f)); cfg = json.load(open(f'checkpoints/{d}/config.json'))
        rows.append({'run': d, 'model': r['model'], 'params': r['n_params'], 'epochs': cfg['epochs'],
                     'autoaug': cfg['autoaugment'], 'mix_prob': cfg['mix_prob'], 'ls': cfg['label_smoothing'],
                     'test_acc': r['test_acc'], 'test_acc_tta': r['test_acc_tta']})
pd.DataFrame(rows).sort_values('test_acc_tta', ascending=False)

## 7. Kaggle submission

In [ ]:
!python predict.py --ckpts {RUN_DIR}/best.pt --test_csv data/test.csv --test_dir data/test_images --out submission.csv
pd.read_csv('submission.csv').head()